REEFPRINT extended-dice: seed42, 24 epochs x192 patches, CE+Dice. Matched pair differs only in training crop policy; validation uses unchanged full native sections. Training-only half-size context (small-grain variant) is experimental, not super-resolution. Test not evaluated; no deployment.


In [ ]:
from pathlib import Path, PurePosixPath
from urllib.parse import quote
from urllib.request import urlopen
from zipfile import ZipFile
import hashlib, json, uuid

publisher_share = 'https://disk.360.yandex.ru/d/wYK_5JyQy0pIcg'
api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key=' + quote(publisher_share, safe='')
archive = Path('/kaggle/working/S2_v2.zip')
expected_sha256 = '64aebd108b3306a8b2e83a945c01f2779bbfd1abeb187b1e2cedcac96cf40c1d'
with urlopen(api, timeout=60) as response:
    download_url = json.load(response)['href']
sha = hashlib.sha256()
size = 0
with urlopen(download_url, timeout=180) as source, archive.open('wb') as target:
    while True:
        chunk = source.read(8 * 1024 * 1024)
        if not chunk:
            break
        target.write(chunk)
        sha.update(chunk)
        size += len(chunk)
        if size % (64 * 1024 * 1024) < len(chunk):
            print(f'Downloaded {size // (1024*1024)} MiB', flush=True)
archive_sha256 = sha.hexdigest()
if archive_sha256 != expected_sha256:
    raise RuntimeError(f'Publisher archive SHA-256 mismatch: {archive_sha256}')
print('Publisher ZIP bytes:', size, '| SHA-256:', archive_sha256)

extract_base = Path('/kaggle/working') / f's2-v2-source-{uuid.uuid4().hex[:8]}'
extract_base.mkdir(parents=True, exist_ok=False)
with ZipFile(archive) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f'Archive CRC check failed at {bad}')
    for member in zf.infolist():
        path = PurePosixPath(member.filename)
        if path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Refusing unsafe archive path: {member.filename}')
    zf.extractall(extract_base)

roots = []
for train_dir in extract_base.rglob('imgs/train'):
    candidate = train_dir.parent.parent
    if (candidate / 'imgs/test').is_dir() and (candidate / 'masks/train').is_dir() and (candidate / 'masks/test').is_dir():
        train_imgs = {p.stem for p in (candidate / 'imgs/train').glob('*.jpg')}
        test_imgs = {p.stem for p in (candidate / 'imgs/test').glob('*.jpg')}
        train_masks = {p.stem for p in (candidate / 'masks/train').glob('*.png')}
        test_masks = {p.stem for p in (candidate / 'masks/test').glob('*.png')}
        if len(train_imgs) == 37 and len(test_imgs) == 12 and train_imgs == train_masks and test_imgs == test_masks:
            roots.append(candidate)
if len(roots) != 1:
    raise RuntimeError(f'Expected exactly one S2-v2 root with 37/12 paired image-mask files; found {len(roots)}. Stop; do not proceed with a different subset.')
s2_root = roots[0].resolve()
print('Verified S2 v2 path:', s2_root)
print('Train pairs:', 37, '| held-out test pairs:', 12)
print('No test images will be used for training or checkpoint selection.')


In [ ]:
import base64, hashlib, io, json, time, zipfile
from pathlib import Path

branch = 'codex/khanya-build-plan'
git_sha = 'f463864d8c81ec77b8ed0381781ba37c81ca4a61'
source_bundle_sha256 = '4cffd60c88fea1a7553f8f154040cf2ce93f7312f0571f3dc6784b5938988528'
source_bundle_b64 = """..."""
run_id = '20260930-s2-seed42-extended-dice-v1'
repo = Path('/kaggle/working') / f'khanya-{run_id}'
repo.mkdir(parents=True, exist_ok=False)
bundle = base64.b64decode(source_bundle_b64, validate=True)
if hashlib.sha256(bundle).hexdigest() != source_bundle_sha256:
    raise RuntimeError('Embedded source bundle SHA-256 mismatch; stop without training.')
with zipfile.ZipFile(io.BytesIO(bundle)) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f'Embedded source ZIP CRC check failed at {bad}')
    for member in zf.infolist():
        if member.filename.startswith('/') or '..' in Path(member.filename).parts:
            raise RuntimeError(f'Unsafe embedded source path: {member.filename}')
    zf.extractall(repo)
source_hashes = {}
for path in sorted((repo / 'src').rglob('*.py')):
    source_hashes[str(path.relative_to(repo)).replace('\\', '/')] = hashlib.sha256(path.read_bytes()).hexdigest()
print('Run ID:', run_id, '| source branch:', branch, '| commit:', git_sha)
print('Embedded source bundle SHA-256:', source_bundle_sha256)
print('Training source modules:', len(source_hashes))
print('Materialized code root:', repo)
data_link = repo / 'data/raw/lumenstone/S2_v2'
data_link.parent.mkdir(parents=True, exist_ok=True)
data_link.symlink_to(s2_root, target_is_directory=True)
train_path = data_link / 'imgs/train'
test_path = data_link / 'imgs/test'
if len(list(train_path.glob('*.jpg'))) != 37 or len(list(test_path.glob('*.jpg'))) != 12:
    raise RuntimeError(f'S2 link is not readable at trainer path: {data_link}')
print('Verified trainer data link:', data_link, '->', s2_root)


In [ ]:
import os, sys, torch, torchvision

os.chdir(repo)
sys.path.insert(0, str(repo))
os.environ['KHANYA_SUBSET'] = 'S2'
os.environ['KHANYA_SEED'] = '42'
os.environ['KHANYA_RUN_ID'] = run_id
os.environ['KHANYA_EPOCHS'] = '24'
os.environ['KHANYA_PATCHES_PER_EPOCH'] = '192'
os.environ['KHANYA_VAL_PATCHES'] = '32'
print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Enable GPU in Kaggle notebook settings, reconnect, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f'GPU memory free/total: {free_bytes/2**30:.1f}/{total_bytes/2**30:.1f} GiB')
print('Default training budget: 24 epochs, 192 train patches/epoch, 32 validation patches, 512px patches, batch 2, AdamW lr 2e-4.')


REEFPRINT extended-dice: seed42, 24 epochs x192 patches, CE+Dice. Matched pair differs only in training crop policy; validation uses unchanged full native sections. Training-only half-size context (small-grain variant) is experimental, not super-resolution. Test not evaluated; no deployment.


In [ ]:
import subprocess, sys
from pathlib import Path

ckpt_dir = repo / 'checkpoints' / f'lumenstone_s2_patches_{run_id}_dice'
if ckpt_dir.exists():
    raise RuntimeError(f'Run checkpoint directory already exists at {ckpt_dir}. Generate a fresh run ID.')
log_path = Path('/kaggle/working') / f'khanya-s2-train-{run_id}.log'
with log_path.open('w') as log_file:
    result = subprocess.run([sys.executable, '-m', 'src.segmentation.train_patches', '--loss', 'dice'],
                            cwd=repo, env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT, text=True)
print(log_path.read_text(errors='replace')[-12000:])
if result.returncode != 0 or not (ckpt_dir / 'best.pt').is_file():
    raise RuntimeError('Training did not produce best.pt. Preserve the log and fix the stated failure before evaluation.')
if not (ckpt_dir / 'validation_history.json').is_file():
    raise RuntimeError('Missing full-section validation history; do not package this run.')
print('Candidate checkpoint:', ckpt_dir / 'best.pt')


In [ ]:
import hashlib, json, zipfile
from pathlib import Path
from src.segmentation import lumenstone as ls
train_ids, validation_ids, test_ids = ls.split_ids()

checkpoint = ckpt_dir / 'best.pt'
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
training_log = log_path.read_text(errors='replace')
history_path = ckpt_dir / 'validation_history.json'
if not history_path.is_file():
    raise RuntimeError('Missing full-section validation history.')
history = json.loads(history_path.read_text())
best_validation = max(history, key=lambda row: row['validation_full_section_metrics']['foreground_macro_iou'])
manifest = {
    'run_id': '20260930-s2-seed42-extended-dice-v1',
    'git_commit': git_sha,
    'code_delivery': 'hash-verified source snapshot embedded by notebook generator',
    'source_bundle_sha256': '4cffd60c88fea1a7553f8f154040cf2ce93f7312f0571f3dc6784b5938988528',
    'source_file_sha256': json.loads("""{"src/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "src/advisor.py": "24777d35b4382f8975c06478936108e23952921d470a42c360bfab2c267c2d45", "src/benchmark.py": "5a0a7e936d58a5c85a5ff3c758dee1fa6bba7687aa9220ba058cbb360e403c01", "src/chromite_pge_falsification.py": "94cb394739463bb0c216b469b1fb733b393ef3481bbf2cb69c4bba34fd407b0d", "src/config.py": "8009746cce6b1b0c5a3b6859d4d375b4eb3c095c470aa5314045fe400cde8d2f", "src/conformal.py": "8fa79aa8a8afb65d5ea2a492d4c1fb00259983bba21b6af6b3e4780a0ba8ef55", "src/data.py": "cdd8f9fd1f361f885f13b80c7e9b2fbd564416e7c53efe6716be8540f288868f", "src/decision_gap.py": "992b49cb37ec86d58f0559de4ea40313855176a571e569b5924c8f4eaaa085b0", "src/evaluate.py": "c5c671c87c6c40997a0f9dcf2c1dd4f9dcd5d8082e13c7558a8e988952d9b1f9", "src/exposure_control.py": "fe7d10cce14a741942afa5330fb213c3686f8f211badeab2c55c9354885f98f2", "src/inspect_pipeline.py": "dd4a26b0b15fd9f93ee93249e70dd2a7fb5432ba51bf7d3a0f18c5254b8f5ab4", "src/modal.py": "efa0596e9c7835c03514df413acb219297faa37013158ecf5d960e840233d8b2", "src/model.py": "2abcc3116eafe09ff98fc3b79ede6acc9a48a477b01e6ba057761aa8436db198", "src/polarimetry.py": "4eaa86e9bd3cbcf6aefc9d0678a91ed7d59066a202e3715c4d6c9b626fb71a7f", "src/preflight.py": "5f2e5a83d9594fd05989aed061f5a83ee55287f7e8217fd559658040051107bb", "src/refinement_audit.py": "1fce9f863e9db099cbd61ce02be51f480d20cf872fa5606a7e462de7407ad7b3", "src/robustness.py": "de030d9a6704470cc2ce74e32d02b7abf07041cd3e353461d529a71ac7fdb52f", "src/s2_section_stats.py": "10e9a6f94194858c1453d4986ff579a19326c52f7c891f24a71f76ff936a8f20", "src/sampling_error.py": "96ad04ffebe8ee0e213a9dc5fae7ce1fa9e066760a994526517da2a060bffd0a", "src/segmentation/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "src/segmentation/baselines.py": "ea9c1834551448aeb1bfb54beba24e3631fab9433f22c939b5b6efb201c19da0", "src/segmentation/config.py": "2ea385edbb3414de681bf0aaa54321495e23538b30292c77be21ea5e18181056", "src/segmentation/data.py": "404d63dc9f2e041144a051c7ef48c544d45c17243a27d16f9305acc78ba052f3", "src/segmentation/evaluate.py": "2f728432e7303fc8892c3ebe5738f9caef682072d82ed7f090a9011f799a4d64", "src/segmentation/latency_benchmark.py": "da8592c8d9eb3179e579a00ed17408f6286f280210e3b49e5b6b874bb2121bd9", "src/segmentation/losses.py": "7c79986e5d4d46a7715a730dc885e93cd544e45cd21fa51f4605c131fdaf308e", "src/segmentation/lumenstone.py": "bec3ae186358fee52152bf5062110186ddf2dbac9ad54c3201816f74ecc10654", "src/segmentation/metrics.py": "799a0c62181164dbf24befb57aaa10588a2bd392c797aa559e68aa30ef7f33c2", "src/segmentation/model.py": "2cead0f9daa832c9d6eeee9f6fb634b43909a188ca8969cd985d0eff65a575de", "src/segmentation/patches.py": "98b59779d1c30c8902c62595d7f414f0e02cc0232a54c597cdcb60a9f5338e09", "src/segmentation/train.py": "022efe6d0a7d640ede292fce16533e37b51b211a5a41d44c952643688e1501fb", "src/segmentation/train_lumenstone.py": "144911df5a5cf93731b54050e83e29c03600eb47d36927a69648a8f8181ffcec", "src/segmentation/train_patches.py": "f1323cf7edc20932112efb259dd468aa28626eed8c73d75aa7720bfb86ebf6b8", "src/train.py": "caab5a4242b64d062413e9be95438da9d7e8221f6e0165f753788652704d9c1d", "src/v1_consistency.py": "7e8ccae8809402769da6c472580d074a5bee32ff1d520919bb9081b82daed5f7"}"""),
    'dataset': 'LumenStone S2 v2',
    'dataset_archive_sha256': archive_sha256,
    'dataset_archive_bytes': size,
    'dataset_source': 'https://imaging.cs.msu.ru/en/research/geology/lumenstone',
    'train_pairs': len(train_ids),
    'source_train_side_pairs': 37,
    'comparison_reference_run_id': '20260929T235548Z-s2-seed42-fullval-v1',
    'validation_pairs': 6,
    'test_pairs': 12,
    'loss': 'cross-entropy + soft Dice; extended matched crop experiment',
    'selection_metric': 'full-section validation foreground macro IoU',
    'best_validation_foreground_macro_iou': best_validation['validation_full_section_metrics']['foreground_macro_iou'],
    'best_epoch': best_validation['epoch'],
    'validation_note': 'Full-resolution predictions on the fixed six validation sections; patch validation is diagnostic only.',
    'validation_history': history,
    'test_used_for_selection': False,
    'test_evaluation_performed': False,
    'test_metrics': None,
    'subset': 'S2',
    'split_seed': ls.SEED,
    'train_image_ids': train_ids,
    'validation_image_ids': validation_ids,
    'test_image_ids': test_ids,
    'training_budget': {'epochs': 24, 'patches_per_epoch': 192, 'validation_patches': 32, 'patch_px': 512, 'batch_size': 2, 'learning_rate': 0.0002},
    'seed': 42,
    'stochasticity_note': 'Matched single-seed CE control; no stable gain claim without repeats.',
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'checkpoint_sha256': digest,
    'rights_note': 'Private research run. Do not publish dataset or derived checkpoint without rights review.'
}
manifest_path = Path('/kaggle/working') / f'khanya-s2-dice-manifest-{run_id}.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
bundle_path = Path('/kaggle/working') / f'khanya-s2-dice-validation-{run_id}.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    archive.write(checkpoint, f'checkpoints/lumenstone_s2_patches_{run_id}_dice/best.pt')
    archive.write(log_path, 'logs/training.log')
    archive.write(history_path, 'validation_history.json')
    archive.write(manifest_path, 'run_manifest.json')
print('Checkpoint SHA256:', digest)
print('Best full-section validation foreground macro IoU:', manifest['best_validation_foreground_macro_iou'])
print('Held-out test split: SEALED; no test metrics produced.')
print('Private run bundle:', bundle_path, f'({bundle_path.stat().st_size/2**20:.1f} MiB)')
